In [2]:
import pandas as pd

# Cargar todas las tablas
customers = pd.read_csv('../data/olist_customers_dataset.csv')
geolocation = pd.read_csv('../data/olist_geolocation_dataset.csv')
order_items = pd.read_csv('../data/olist_order_items_dataset.csv')
payments = pd.read_csv('../data/olist_order_payments_dataset.csv')
reviews = pd.read_csv('../data/olist_order_reviews_dataset.csv')
orders = pd.read_csv('../data/olist_orders_dataset.csv')
products = pd.read_csv('../data/olist_products_dataset.csv')
sellers = pd.read_csv('../data/olist_sellers_dataset.csv')
category_translation = pd.read_csv('../data/product_category_name_translation.csv')

# Diccionario para iterar fácilmente
tablas = {
    'customers': customers,
    'geolocation': geolocation,
    'order_items': order_items,
    'payments': payments,
    'reviews': reviews,
    'orders': orders,
    'products': products,
    'sellers': sellers,
    'category_translation': category_translation
}

for nombre, df in tablas.items():
    print(f"{nombre}: {df.shape[0]} filas, {df.shape[1]} columnas")

customers: 99441 filas, 5 columnas
geolocation: 1000163 filas, 5 columnas
order_items: 112650 filas, 7 columnas
payments: 103886 filas, 5 columnas
reviews: 99224 filas, 7 columnas
orders: 99441 filas, 8 columnas
products: 32951 filas, 9 columnas
sellers: 3095 filas, 4 columnas
category_translation: 71 filas, 2 columnas


In [3]:
for nombre, df in tablas.items():
    print(f"\n{'='*50}")
    print(f"TABLA: {nombre}")
    print(f"{'='*50}")
    print(df.dtypes)
    print("\nValores nulos por columna:")
    nulos = df.isnull().sum()
    print(nulos[nulos > 0] if nulos.sum() > 0 else "Sin valores nulos")


TABLA: customers
customer_id                 object
customer_unique_id          object
customer_zip_code_prefix     int64
customer_city               object
customer_state              object
dtype: object

Valores nulos por columna:
Sin valores nulos

TABLA: geolocation
geolocation_zip_code_prefix      int64
geolocation_lat                float64
geolocation_lng                float64
geolocation_city                object
geolocation_state               object
dtype: object

Valores nulos por columna:
Sin valores nulos

TABLA: order_items
order_id                object
order_item_id            int64
product_id              object
seller_id               object
shipping_limit_date     object
price                  float64
freight_value          float64
dtype: object

Valores nulos por columna:
Sin valores nulos

TABLA: payments
order_id                 object
payment_sequential        int64
payment_type             object
payment_installments      int64
payment_value           float6

In [4]:
# ¿Los nulos en fechas de entrega corresponden a pedidos no entregados?
print(orders[orders['order_delivered_customer_date'].isnull()]['order_status'].value_counts())

order_status
shipped        1107
canceled        619
unavailable     609
invoiced        314
processing      301
delivered         8
created           5
approved          2
Name: count, dtype: int64


In [5]:
# Inconsistencia: pedidos "delivered" sin fecha de entrega registrada
inconsistencias = orders[
    (orders['order_status'] == 'delivered') & 
    (orders['order_delivered_customer_date'].isnull())
]
print(f"Pedidos inconsistentes: {len(inconsistencias)}")
inconsistencias[['order_id', 'order_status', 'order_purchase_timestamp', 'order_delivered_customer_date']]

Pedidos inconsistentes: 8


,order_id,order_status,order_purchase_timestamp,order_delivered_customer_date
3002,2d1e2d5bf4dc7227b3bfebb81328c15f,delivered,2017-11-28 17:44:07,NaN
20618,f5dd62b788049ad9fc0526e3ad11a097,delivered,2018-06-20 06:58:43,NaN
43834,2ebdfc4f15f23b91474edf87475f108e,delivered,2018-07-01 17:05:11,NaN
79263,e69f75a717d64fc5ecdfae42b2e8e086,delivered,2018-07-01 22:05:55,NaN
82868,0d3268bad9b086af767785e3f0fc0133,delivered,2018-07-01 21:14:02,NaN
92643,2d858f451373b04fb5c984a1cc2defaf,delivered,2017-05-25 23:22:43,NaN
97647,ab7c89dc1bf4a1ead9d6ec1ec8968a84,delivered,2018-06-08 12:09:39,NaN
98038,20edc82cf5400ce95e1afacc25798b31,delivered,2018-06-27 16:09:12,NaN


## Hallazgo: Inconsistencia en fechas de entrega

8 pedidos (0.008% del total) tienen status "delivered" pero carecen de 
`order_delivered_customer_date`. Notablemente, 3 de estos casos ocurren 
el mismo día (2018-07-01), lo que sugiere un posible problema puntual 
del sistema en esa fecha — aunque no se puede confirmar la causa exacta 
con los datos disponibles. Estos registros se excluirán de análisis que 
dependan específicamente de tiempos de entrega, pero se conservan para 
el resto del análisis.

In [6]:
# ¿Cuánto volumen de ventas representan los productos con categoría nula?
productos_incompletos = products[products['product_category_name'].isnull()]['product_id']
items_afectados = order_items[order_items['product_id'].isin(productos_incompletos)]

print(f"Productos con metadata incompleta: {len(productos_incompletos)}")
print(f"Items de pedidos afectados: {len(items_afectados)}")
print(f"% del total de items: {len(items_afectados) / len(order_items) * 100:.2f}%")

Productos con metadata incompleta: 610
Items de pedidos afectados: 1603
% del total de items: 1.42%


In [7]:
# Imputación de products
products['product_category_name'] = products['product_category_name'].fillna('sem_categoria')
products['product_name_lenght'] = products['product_name_lenght'].fillna(0)
products['product_description_lenght'] = products['product_description_lenght'].fillna(0)
products['product_photos_qty'] = products['product_photos_qty'].fillna(0)

# Los 2 nulos de dimensiones físicas: imputamos con la mediana
for col in ['product_weight_g', 'product_length_cm', 'product_height_cm', 'product_width_cm']:
    products[col] = products[col].fillna(products[col].median())

# Verifica que ya no queden nulos
print(products.isnull().sum().sum())

0


In [8]:
# Empezamos con orders como tabla base
df = orders.merge(order_items, on='order_id', how='left')
df = df.merge(products, on='product_id', how='left')
df = df.merge(customers, on='customer_id', how='left')
df = df.merge(sellers, on='seller_id', how='left')
df = df.merge(category_translation, on='product_category_name', how='left')

# Payments: un pedido puede tener varios pagos (ej. tarjeta + voucher),
# así que agregamos por order_id para no duplicar filas
payments_agg = payments.groupby('order_id').agg(
    payment_value_total=('payment_value', 'sum'),
    payment_type_principal=('payment_type', 'first'),
    payment_installments_max=('payment_installments', 'max')
).reset_index()
df = df.merge(payments_agg, on='order_id', how='left')

# Reviews: un pedido normalmente tiene un solo review, pero por seguridad
# tomamos el primero si hubiera duplicados
reviews_dedup = reviews.drop_duplicates(subset='order_id', keep='first')
df = df.merge(
    reviews_dedup[['order_id', 'review_score']], 
    on='order_id', 
    how='left'
)

print(f"DataFrame maestro: {df.shape[0]} filas, {df.shape[1]} columnas")
df.head()

DataFrame maestro: 113425 filas, 34 columnas


,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date,order_item_id,product_id,...,customer_city,customer_state,seller_zip_code_prefix,seller_city,seller_state,product_category_name_english,payment_value_total,payment_type_principal,payment_installments_max,review_score
0,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2017-10-02 10:56:33,2017-10-02 11:07:15,2017-10-04 19:55:00,2017-10-10 21:25:13,2017-10-18 00:00:00,1.0,87285b34884572647811a353c7ac498a,...,sao paulo,SP,9350.0,maua,SP,housewares,38.71,credit_card,1.0,4.0
1,53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,delivered,2018-07-24 20:41:37,2018-07-26 03:24:27,2018-07-26 14:31:00,2018-08-07 15:27:45,2018-08-13 00:00:00,1.0,595fac2a385ac33a80bd5114aec74eb8,...,barreiras,BA,31570.0,belo horizonte,SP,perfumery,141.46,boleto,1.0,4.0
2,47770eb9100c2d0c44946d9cf07ec65d,41ce2a54c0b03bf3443c3d931a367089,delivered,2018-08-08 08:38:49,2018-08-08 08:55:23,2018-08-08 13:50:00,2018-08-17 18:06:29,2018-09-04 00:00:00,1.0,aa4383b373c6aca5d8797843e5594415,...,vianopolis,GO,14840.0,guariba,SP,auto,179.12,credit_card,3.0,5.0
3,949d5b44dbf5de918fe9c16f97b45f8a,f88197465ea7920adcdbec7375364d82,delivered,2017-11-18 19:28:06,2017-11-18 19:45:59,2017-11-22 13:39:59,2017-12-02 00:28:42,2017-12-15 00:00:00,1.0,d0b61bfb1de832b15ba9d266ca96e5b0,...,sao goncalo do amarante,RN,31842.0,belo horizonte,MG,pet_shop,72.20,credit_card,1.0,5.0
4,ad21c59c0840e6cb83a9ceb5573f8159,8ab97904e6daea8866dbdbc4fb7aad2c,delivered,2018-02-13 21:18:39,2018-02-13 22:20:29,2018-02-14 19:46:34,2018-02-16 18:17:02,2018-02-26 00:00:00,1.0,65266b2da20d04dbe00c5c2d3bb7859e,...,santo andre,SP,8752.0,mogi das cruzes,SP,stationery,28.62,credit_card,1.0,5.0


In [9]:
# Verificar unicidad de las llaves de merge en cada tabla
print("products - product_id únicos vs total filas:", products['product_id'].nunique(), "/", len(products))
print("customers - customer_id únicos vs total filas:", customers['customer_id'].nunique(), "/", len(customers))
print("sellers - seller_id únicos vs total filas:", sellers['seller_id'].nunique(), "/", len(sellers))
print("category_translation - product_category_name únicos vs total filas:", category_translation['product_category_name'].nunique(), "/", len(category_translation))

products - product_id únicos vs total filas: 32951 / 32951
customers - customer_id únicos vs total filas: 99441 / 99441
sellers - seller_id únicos vs total filas: 3095 / 3095
category_translation - product_category_name únicos vs total filas: 71 / 71


In [10]:
print("orders - order_id únicos vs total filas:", orders['order_id'].nunique(), "/", len(orders))
print("order_items - order_id repetidos (normal) pero verificamos función distinta:")
print("order_items total filas:", len(order_items))
print("order_items - combinación order_id + order_item_id única:", 
      order_items.duplicated(subset=['order_id', 'order_item_id']).sum(), "duplicados")

orders - order_id únicos vs total filas: 99441 / 99441
order_items - order_id repetidos (normal) pero verificamos función distinta:
order_items total filas: 112650
order_items - combinación order_id + order_item_id única: 0 duplicados


In [11]:
ordenes_sin_items = set(orders['order_id']) - set(order_items['order_id'])
print(f"Pedidos en 'orders' sin ningún item en 'order_items': {len(ordenes_sin_items)}")

Pedidos en 'orders' sin ningún item en 'order_items': 775


In [13]:
# ¿Qué status tienen los pedidos sin items?
print(orders[orders['order_id'].isin(ordenes_sin_items)]['order_status'].value_counts())

order_status
unavailable    603
canceled       164
created          5
invoiced         2
shipped          1
Name: count, dtype: int64


## Hallazgo: Pedidos sin items asociados

775 pedidos (0.78% del total) no tienen ningún registro en `order_items`. 
Esto se debe a que el merge orders→order_items es 'left join', preservando 
pedidos que probablemente se cancelaron o quedaron indisponibles antes de 
procesarse. Estas filas se conservan en el DataFrame maestro con valores 
nulos en las columnas de producto/precio, y se excluirán de análisis 
específicos de ventas (donde no aplican), pero se mantienen para análisis 
de tasa de cancelación.